# Notebook A: Setup & Production Matching

Loads the mineral inventory and mine-plant links, filters idle mines,
then matches COCHILCO production data to inventory facilities for ALL
Chilean minerals (metallic and non-metallic). Creates USD valuations
using COCHILCO Anuario prices, implied FOB prices, and USGS benchmarks.

**Reads:** backup CSVs (`Chile_Minerals_Inventory_backup.csv`, `Chile_Mine_Plant_Links_backup.csv`)  
**Writes:** `_pipeline_state_0.pkl` (after idle-mine filter); `_pipeline_state_2.pkl` (final, after pricing)  
**CSVs output:** `Chile_Minerals_Inventory.csv` (updated with production/price columns),
`Chile_National_Production_2024.csv` (national summary table, one row per mineral)

**Minerals with facility-level allocation:** Cu, Mo, Au, Ag, Fe, Zn, Pb,
Li (by compound), Iodine, Nitrates, Boron, Potash, Salt, Limestone,
Gypsum, Silica, and ~15 other non-metallic minerals.

**National-level summary:** All ~35 minerals with nonzero 2024 production.


In [ ]:
import sys
import os

# Find the Chile project root (contains scripts/ and inputs/) by walking up from cwd.
def _find_chile_root():
    d = os.path.abspath(os.getcwd())
    for _ in range(5):
        if os.path.isdir(os.path.join(d, "scripts")) and os.path.isdir(os.path.join(d, "inputs")):
            return d
        d = os.path.dirname(d)
    raise RuntimeError(
        "Could not find Chile project root (expected to contain scripts/ and inputs/)."
    )

_utils_dir = os.path.join(_find_chile_root(), "scripts")
if _utils_dir not in sys.path:
    sys.path.insert(0, _utils_dir)

from pipeline_utils import *   # constants, paths, utilities
from pipeline_constants import *  # hard-coded assumptions 

inv_path   = os.path.join(DIR_INTERMED, 'Chile_Minerals_Inventory.csv')
links_path = os.path.join(DIR_INTERMED, 'Chile_Mine_Plant_Links.csv')

for p in [inv_path, links_path]:
    bak = p.replace('.csv', '_backup.csv')
    if os.path.exists(p) and not os.path.exists(bak):
        shutil.copy2(p, bak)
        print(f'Backed up: {bak}')

inv   = pd.read_csv(inv_path.replace('.csv', '_backup.csv'), low_memory=False)
links = pd.read_csv(links_path.replace('.csv', '_backup.csv'))

comm_col = 'COMMODITY_LIST_STR' if 'COMMODITY_LIST_STR' in inv.columns else 'ALL_COMMODITIES_RAW'
print(f'Inventory: {len(inv)} records')
print(f'Links:     {len(links)} records')

# Idle mines would create dangling edges in the supply chain (a mine with no
# active downstream path). Remove their links before edge-building starts.

idle_mines     = set(inv.loc[inv['FACILITY_TYPE'] == 'Mine (idle)', 'FACILITY_NAME'])
idle_link_mask = links['MINE_NAME'].isin(idle_mines)
n_idle_links   = idle_link_mask.sum()
links          = links[~idle_link_mask].reset_index(drop=True)
print(f'Removed {n_idle_links} links from {len(idle_mines)} idle mines')
print(f'Links after idle filter: {len(links)}')


_state = {
    'inv': inv, 'links': links, 'comm_col': comm_col, 'idle_mines': idle_mines,
    'inv_path': inv_path, 'links_path': links_path,
    'COMPANY_TO_DEPOSIT': COMPANY_TO_DEPOSIT,
    'CODELCO_EXTRA_SEARCH': CODELCO_EXTRA_SEARCH,
    'SMELTERS': SMELTERS, 'PORTS': PORTS,
    'SMELTER_NAME_MAP': SMELTER_NAME_MAP,
    'DEDICATED_PORT': DEDICATED_PORT,
    'CODELCO_CATHODE_ROUTING': CODELCO_CATHODE_ROUTING,
    'MATCH_DISAMBIGUATION': MATCH_DISAMBIGUATION,
    'IRON_MINE_NAMES': IRON_MINE_NAMES,
    'ZINC_MINE_NAMES': ZINC_MINE_NAMES,
    'n_idle_links': n_idle_links,
}
save_state(_state, 0)


Inventory: 461 records
Links:     1370 records
Removed 263 links from 80 idle mines
Links after idle filter: 1107
State saved to /Users/leoss/Desktop/Website-/Portfolio/Website-/projects/Chile/output/intermediary/_pipeline_state_0.pkl


## 2A-2D: Copper, Molybdenum & Lithium Production Matching

In [ ]:

# 2A. Codelco division mapping
# COCHILCO reports Codelco output by division (e.g. "División El Teniente"),
# while the USGS/SNL inventory uses individual mine names ("El Teniente mine").
# This step links the two so Codelco production flows to the right inventory rows.

section_header("2A. CODELCO DIVISION MAPPING")

codelco_divisions = {k: v for k, v in COMPANY_TO_DEPOSIT.items() if k.startswith("División")}
for div_name, terms in codelco_divisions.items():
    extra = CODELCO_EXTRA_SEARCH.get(div_name, [])
    matched_idx = search_inventory(inv, terms + extra)
    if matched_idx:
        print(f"  {div_name}")
        for idx in matched_idx:
            if pd.isna(inv.at[idx, "OPERATOR_NAME"]) or inv.at[idx, "OPERATOR_NAME"] == "":
                inv.at[idx, "OPERATOR_NAME"] = "Codelco"
            print(f"    -> {inv.at[idx, 'FACILITY_NAME']} ({inv.at[idx, 'FACILITY_TYPE']})")
    else:
        print(f"  WARNING: {div_name} -> NO MATCH IN INVENTORY")

# 2B. Copper production matching

section_header("2B. COPPER PRODUCTION MATCHING")

nat = pd.read_excel(COCHILCO_PATH, sheet_name="A_National_Production", header=3, index_col=0)
nat.columns = [int(c) if isinstance(c, (int, float)) else c for c in nat.columns]
latest_yr = max(c for c in nat.columns if isinstance(c, int))

commodity_search = {
    "Copper": "COBRE.*Miles de TM", "Molybdenum": "MOLIBDENO.*TM de fino",
    "Gold": "ORO.*Kg de fino", "Silver": "PLATA.*Kg de fino",
    "Iron": "HIERRO.*Miles de TM", "Zinc": "ZINC.*TM de fino",
}
print(f"COCHILCO latest year: {latest_yr}")
for comm, pattern in commodity_search.items():
    match = [r for r in nat.index if re.search(pattern, str(r), re.IGNORECASE)]
    if match:
        print(f"  {comm:<15} {nat.loc[match[0], latest_yr]:>15,.1f}")

cu_co = pd.read_excel(COCHILCO_PATH, sheet_name="B1_Copper_by_Company", header=3, index_col=0)
cu_co.columns = [int(c) if isinstance(c, (int, float)) else c for c in cu_co.columns]

# Idempotent column creation
for col, default in [("COCHILCO_CU_2024_KMT", np.nan), ("COCHILCO_COMPANY", "")]:
    if col not in inv.columns:
        inv[col] = default

cu_matched, cu_matched_prod = 0, 0.0
for company, search_terms in COMPANY_TO_DEPOSIT.items():
    if company not in cu_co.index or latest_yr not in cu_co.columns:
        continue
    prod = cu_co.loc[company, latest_yr]
    if not isinstance(prod, (int, float)) or pd.isna(prod):
        continue
    for term in search_terms:
        mask = inv["FACILITY_NAME"].str.contains(term, case=False, na=False, regex=False)
        mine_mask = mask & inv["FACILITY_TYPE"].str.contains("Mine", case=False, na=False)
        matches = inv[mine_mask] if mine_mask.any() else inv[mask]
        if len(matches) > 0:
            # Disambiguate when multiple mines match the same term
            if company in MATCH_DISAMBIGUATION and len(matches) > 1:
                preferred = MATCH_DISAMBIGUATION[company]
                exact = matches[matches["FACILITY_NAME"] == preferred]
                if len(exact) > 0:
                    matches = exact
            idx = matches.index[0]
            if pd.isna(inv.at[idx, "COCHILCO_CU_2024_KMT"]):
                inv.at[idx, "COCHILCO_CU_2024_KMT"] = prod
                inv.at[idx, "COCHILCO_COMPANY"] = company
                cu_matched += 1
                cu_matched_prod += prod
                print(f"  {company:<35} -> {inv.at[idx, 'FACILITY_NAME']:<35} {prod:>8,.1f} kMT")
            break
    else:
        print(f"  WARNING: {company:<35} -> NOT MATCHED ({prod:,.1f} kMT)")

# Dynamic total (no hardcoded fallback)
cu_total = cu_co.loc["TOTAL", latest_yr] if "TOTAL" in cu_co.index and latest_yr in cu_co.columns else cu_matched_prod
if cu_total == cu_matched_prod and "TOTAL" not in cu_co.index:
    print(f"  WARNING: 'TOTAL' row not found. Using sum of matched: {cu_total:,.1f} kMT.")
print(f"\nCu matched: {cu_matched} companies, {cu_matched_prod:,.1f} / {cu_total:,.1f} kMT "
      f"({cu_matched_prod/cu_total*100:.1f}%)")

# 2C. Molybdenum production 

section_header("2C. MOLYBDENUM PRODUCTION MATCHING")

if "COCHILCO_MO_2024_MT" not in inv.columns:
    inv["COCHILCO_MO_2024_MT"] = np.nan

if os.path.exists(COCHILCO_ORIG):
    wb_orig = openpyxl.load_workbook(COCHILCO_ORIG, read_only=True, data_only=True)
    ws = wb_orig["Tabla 4.2"]
    rows_raw = list(ws.iter_rows(values_only=True))

    yr_row_idx, year_cols = None, {}
    for i, row in enumerate(rows_raw):
        if sum(1 for v in row if isinstance(v, (int, float)) and 2004 < v < 2035) >= 8:
            yr_row_idx = i
            year_cols = {int(v): j for j, v in enumerate(row) if isinstance(v, (int, float)) and 2004 < v < 2035}
            break

    mo_by_company, current_company = {}, None
    for i in range(yr_row_idx + 1, len(rows_raw)):
        row = rows_raw[i]
        label = str(row[0]).strip() if row[0] is not None else ""
        if not label or label.startswith("(") or label.startswith("Fuente"):
            continue
        has_data = any(isinstance(row[j], (int, float)) and row[j] != 0
                       for j in year_cols.values() if j < len(row))
        is_sub = "CONCENTRADO" in label or "ÓXIDO" in label

        if has_data and not is_sub:
            vals = {yr: row[col] for yr, col in year_cols.items() if col < len(row) and isinstance(row[col], (int, float))}
            mo_by_company[label] = vals
            current_company = None
        elif not has_data and not is_sub:
            current_company = label
        elif has_data and current_company:
            vals = {yr: row[col] for yr, col in year_cols.items() if col < len(row) and isinstance(row[col], (int, float))}
            if current_company not in mo_by_company:
                mo_by_company[current_company] = {}
            for yr, v in vals.items():
                mo_by_company[current_company][yr] = mo_by_company[current_company].get(yr, 0) + v

    # COCHILCO Tabla 4.2 uses division names for Codelco and company names for
    # private operators. The USGS inventory uses mine names. MO_MAP translates
    # between the two so Mo production can be attached to the right inventory row.
    MO_MAP = {
        "Divisiones Chuquicamata y Radomiro Tomic": ["Chuquicamata", "Radomiro Tomic"],
        "División Salvador": ["Salvador"], "División Andina": ["Andina"],
        "División El Teniente": ["El Teniente", "Teniente"],
        "Collahuasi": ["Collahuasi"], "Sierra Gorda": ["Sierra Gorda"],
        "Caserones": ["Caserones"], "Los Pelambres": ["Los Pelambres", "Pelambres"],
        "Anglo American Sur": ["Los Bronces", "Bronces"],
        "Centinela": ["Centinela"], "Spence": ["Spence"],
        "Quebrada Blanca": ["Quebrada Blanca"],
        "Valle Central": ["Valle Central"], 
    }

    mo_latest_yr = max(year_cols.keys()) if year_cols else 2024
    print(f"  Mo latest year: {mo_latest_yr}")

    mo_matched, mo_matched_prod = 0, 0.0
    for company, search_terms in MO_MAP.items():
        if company not in mo_by_company:
            continue
        prod = mo_by_company[company].get(mo_latest_yr, 0)
        if prod <= 0:
            continue
        # Handle split allocations for combined entries
        if company in MO_SPLIT:
            for mine_term, share in MO_SPLIT[company]:
                sub_prod = prod * share
                sub_idx = search_inventory(inv, [mine_term], require_mine=True)
                if sub_idx:
                    idx = sub_idx[0]
                    if pd.isna(inv.at[idx, "COCHILCO_MO_2024_MT"]):
                        inv.at[idx, "COCHILCO_MO_2024_MT"] = sub_prod
                        mo_matched += 1
                        mo_matched_prod += sub_prod
                        print(f"  {company:<50} -> {inv.at[idx, 'FACILITY_NAME']:<30} {sub_prod:>10,.1f} MT ({share*100:.0f}%)")
                else:
                    print(f"  WARNING: {company} / {mine_term:<30} -> NOT MATCHED ({sub_prod:,.1f} MT)")
        else:
            matched_idx = search_inventory(inv, search_terms, require_mine=True)
            if matched_idx:
                idx = matched_idx[0]
                if pd.isna(inv.at[idx, "COCHILCO_MO_2024_MT"]):
                    inv.at[idx, "COCHILCO_MO_2024_MT"] = prod
                    mo_matched += 1
                    mo_matched_prod += prod
                    print(f"  {company:<50} -> {inv.at[idx, 'FACILITY_NAME']:<30} {prod:>10,.1f} MT")
            else:
                print(f"  WARNING: {company:<50} -> NOT MATCHED ({prod:,.1f} MT)")

    mo_national = mo_by_company.get("TOTAL", {}).get(mo_latest_yr, mo_matched_prod)
    if mo_national <= 0:
        mo_national = mo_matched_prod
    print(f"\nMo matched: {mo_matched}, {mo_matched_prod:,.1f} / {mo_national:,.1f} MT "
          f"({mo_matched_prod/mo_national*100:.1f}%)")
    wb_orig.close()
else:
    print(f"  Original COCHILCO not found at {COCHILCO_ORIG}, skipping Mo parsing")

# 2D. Lithium: commodity fixes and link rebuild
# Lithium needs two corrections before supply chain edges are built:
#  1. USGS classifies some Atacama salar facilities as Potash because K and Li
#     co-occur in the brine. Add Lithium to their commodity list so they are
#     included in lithium matching.
#  2. The original mine-plant links use a distance threshold that is too tight
#     for the Atacama Basin (Salar de Atacama to Salar del Carmen is ~205 km).
#     Rebuild links from scratch with calibrated caps.

section_header("2D. LITHIUM FIXES AND LINK CLEANUP")

potash_li = inv[
    inv["SOURCE"].str.contains("USGS", na=False) &
    inv["PRIMARY_COMMODITY"].str.contains("Potash", case=False, na=False) &
    inv["FACILITY_NAME"].str.contains("Salar|Carmen|Atacama", case=False, na=False)
]
li_fixes = sum(add_commodity(idx, "Lithium", inv, comm_col) for idx in potash_li.index)
for idx in inv[(inv["FACILITY_NAME"] == "Salar de Atacama") &
               inv["FACILITY_TYPE"].str.contains("Mine", case=False, na=False)].index:
    add_commodity(idx, "Potassium", inv, comm_col)
print(f"  Lithium commodity fixes: {li_fixes} records")

STAGE_LI = {"Mine (active)": "extraction", "Mine (idle)": "extraction_idle",
            "Prospect/Project": "extraction", "Mine (USGS)": "extraction"}
inv["_stage"] = inv["FACILITY_TYPE"].map(STAGE_LI)

li_mines = inv[(inv["_stage"] == "extraction") &
               inv[comm_col].str.contains("Lithium", case=False, na=False) &
               inv["LATITUD"].notna()]
li_plants = inv[(inv["_stage"].isna()) &  # non-extraction = processing
                inv[comm_col].str.contains("Lithium", case=False, na=False) &
                inv["LATITUD"].notna()]

# Deduplicate co-located plants before linking (e.g. triple Chemetall Foote entries)
li_plants_dedup = li_plants.copy()
li_plants_dedup["_loc_key"] = (
    li_plants_dedup["LATITUD"].round(2).astype(str) + "_" +
    li_plants_dedup["LONGITUD"].round(2).astype(str)
)
li_plants_dedup = li_plants_dedup.drop_duplicates(subset=["_loc_key"], keep="first")
li_plants_dedup = li_plants_dedup.drop(columns=["_loc_key"])

# Distance caps: 210 km for active mines (Salar de Atacama to Salar del
# Carmen plants is ~200-207 km), 80 km for prospects
new_li_links = []
for _, mrow in li_mines.iterrows():
    is_prospect = "Prospect" in str(mrow["FACILITY_TYPE"])
    max_dist = 80 if is_prospect else 210
    for _, prow in li_plants_dedup.iterrows():
        dist = haversine_km(mrow["LATITUD"], mrow["LONGITUD"], prow["LATITUD"], prow["LONGITUD"])
        if dist <= max_dist:
            new_li_links.append({
                "MINE_NAME": mrow["FACILITY_NAME"], "MINE_TYPE": mrow["FACILITY_TYPE"],
                "MINE_STATUS": mrow.get("STATUS", ""), "MINE_LAT": mrow["LATITUD"],
                "MINE_LON": mrow["LONGITUD"], "MINE_REGION": mrow.get("REGION", ""),
                "MINE_OPERATOR": mrow.get("OPERATOR_NAME", ""),
                "PLANT_NAME": prow["FACILITY_NAME"], "PLANT_TYPE": prow["FACILITY_TYPE"],
                "PLANT_STATUS": prow.get("STATUS", ""), "PLANT_LAT": prow["LATITUD"],
                "PLANT_LON": prow["LONGITUD"], "PLANT_OPERATOR": prow.get("OPERATOR_NAME", ""),
                "PLANT_OWNER": prow.get("OWNER_NAME", ""),
                "PLANT_CAPACITY": prow.get("CAPACITY"),
                "PLANT_CAPACITY_UNITS": prow.get("CAPACITY_UNITS", ""),
                "SHARED_COMMODITIES": "Lithium", "DISTANCE_KM": round(dist, 1),
            })

old_li = len(links[links["SHARED_COMMODITIES"].str.contains("Lithium", na=False)])
links = links[~links["SHARED_COMMODITIES"].str.contains("Lithium", na=False)]

if new_li_links:
    li_df = pd.DataFrame(new_li_links)
    for col in links.columns:
        if col not in li_df.columns:
            li_df[col] = np.nan
    links = pd.concat([links, li_df[links.columns]], ignore_index=True)

li_mask = links["SHARED_COMMODITIES"].str.contains("Lithium", na=False)
# Prospect distance already handled above (80 km cap)
li_subset = links[li_mask].copy()
non_li = links[~links["SHARED_COMMODITIES"].str.contains("Lithium", na=False)]
li_subset["_loc"] = li_subset["PLANT_LAT"].round(2).astype(str) + "_" + li_subset["PLANT_LON"].round(2).astype(str)
li_deduped = li_subset.sort_values("DISTANCE_KM").drop_duplicates(subset=["MINE_NAME", "_loc"], keep="first").drop(columns=["_loc"])
links = pd.concat([non_li, li_deduped], ignore_index=True).sort_values(["MINE_NAME", "DISTANCE_KM"])

inv.drop(columns=["_stage"], inplace=True, errors="ignore")
li_final = len(links[links["SHARED_COMMODITIES"].str.contains("Lithium", na=False)])
print(f"  Lithium links: {old_li} old -> {li_final} cleaned")


2A. CODELCO DIVISION MAPPING
  División El Teniente
    -> El Teniente (Mine (active))
    -> El Teniente plant (Processing Plant)
  División Chuquicamata
    -> Chuquicamata (Mine (active))
    -> Chuquicamata Division (Processing Plant)
    -> Chuquicamata Mine (plant to acid-leach fine copper) (Processing Plant)
    -> Chuquicamata plant (Processing Plant)
    -> Chuquicamata SX-EW plant (oxide) and smelter (Smelter)
  División Radomiro Tomic
    -> Radomiro Tomic (Mine (active))
    -> Radomiro Tomic SX-EW plant (SX-EW Plant)
  División Andina
    -> Andina (Mine (active))
  División Ministro Hales
    -> Ministro Hales (Mine (active))
  División Gabriela Mistral
    -> Gabriela Mistral (Mine (active))
    -> Gabriela Mistral SX-EW plant (SX-EW Plant)
  División Salvador
    -> Salvador (Mine (active))
    -> Potrerillos plant (Processing Plant)
    -> Potrerillos SX-EW refinery and smelter (Smelter)

2B. COPPER PRODUCTION MATCHING
COCHILCO latest year: 2024
  Copper              

## 2E: Extended Mineral Production Matching

Matches ALL minerals from COCHILCO C_* regional production sheets to
inventory facilities. Metallic minerals (Au, Ag, Fe, Zn, Pb) use
resource/reserve weighting. Non-metallic minerals (Li compounds, Iodine,
Nitrates, Boron, Salt, Limestone, Gypsum, etc.) use equal distribution
among matching facilities in each producing region.

Aggregate categories (Lithium Compounds, Calcium Carbonate, Clay, Silica
Ores) are NOT matched here to avoid double-counting with subcategories.
They appear in the national summary table (Cell 2G).


In [ ]:

# 2E. EXTENDED MINERAL PRODUCTION MATCHING
#
# Extends the original Au/Ag/Fe/Zn matching to ALL minerals reported in
# COCHILCO C_* regional production sheets. This covers:
#   - Metallic: Au, Ag, Fe, Zn, Pb  (unit from label: TM/Kg/kMT)
#   - Non-metallic: Li, I, Nitrates, Boron, Salt, CaCO3, Gypsum, etc.
#     (all in MT, as stated in the section header)
#
# Allocation strategy per mineral:
#   "resource_reserve" — existing approach using Sernageomin reserve/
#                        resource tonnage weights (Au, Ag, Fe, Zn)
#   "equal"           — equal distribution among matching active
#                        facilities in the region (non-metallics)
#   "national_only"   — no facility allocation attempted; national
#                        total stored in summary table only
#
# Weighting priority for resource_reserve minerals:
#   1. Resource tonnage (e.g. Gold_Resource)
#   2. Reserve tonnage (e.g. Gold_Reserve)
#   Mines with neither receive weight 0 and are excluded.

section_header("2E. EXTENDED MINERAL PRODUCTION MATCHING")

# Mineral definitions
#
# Each entry defines:
#   col            — inventory column for matched production
#   unit           — target unit for the column
#   pattern        — COCHILCO row label pattern (Spanish, uppercase match)
#   commodity_kw   — keyword to match/add in COMMODITY_LIST_STR
#   resource_col   — inventory column for resource tonnage (or None)
#   reserve_col    — inventory column for reserve tonnage (or None)
#   allocation     — "resource_reserve", "equal", or "national_only"
#   is_metallic    — True if unit must be parsed from label; False = MT
#   facility_types — regex for FACILITY_TYPE filter (default: Mine.*active)

MINERAL_COLS = {
    # Metallic minerals (unit detected from label)
    "Gold": {
        "col": "COCHILCO_AU_2024_KG", "unit": "Kg",
        "pattern": "ORO", "commodity_kw": "Gold",
        "resource_col": "Gold_Resource", "reserve_col": "Gold_Reserve",
        "allocation": "resource_reserve", "is_metallic": True,
    },
    "Silver": {
        "col": "COCHILCO_AG_2024_KG", "unit": "Kg",
        "pattern": "PLATA", "commodity_kw": "Silver",
        "resource_col": "Silver_Resource", "reserve_col": "Silver_Reserve",
        "allocation": "resource_reserve", "is_metallic": True,
    },
    "Iron": {
        "col": "COCHILCO_FE_2024_KMT", "unit": "kMT",
        "pattern": "HIERRO", "commodity_kw": "Iron",
        "resource_col": "Iron_Resource", "reserve_col": "Iron_Reserve",
        "allocation": "resource_reserve", "is_metallic": True,
    },
    "Zinc": {
        "col": "COCHILCO_ZN_2024_MT", "unit": "MT",
        "pattern": "ZINC", "commodity_kw": "Zinc",
        "resource_col": "Zinc_Resource", "reserve_col": "Zinc_Reserve",
        "allocation": "resource_reserve", "is_metallic": True,
    },
    "Lead": {
        "col": "COCHILCO_PB_2024_MT", "unit": "MT",
        "pattern": "PLOMO", "commodity_kw": "Lead",
        "resource_col": "Lead_Resource", "reserve_col": "Lead_Reserve",
        "allocation": "resource_reserve", "is_metallic": True,
    },
    # Non-metallic minerals (all MT per section header)
    # High-value exported minerals
    # NOTE: Aggregate category entries (e.g. Lithium Compounds, Calcium Carbonate,
    # Nitrates, Clay) are NOT matched here to avoid double-counting with their
    # sub-component rows. The authoritative list is DOUBLE_COUNT_EXCLUDE in
    # pipeline_constants.py — add entries there if COCHILCO adds new aggregates.
    "Lithium_Carbonate": {
        "col": "COCHILCO_LICO3_2024_MT", "unit": "MT",
        "pattern": "CARBONATO DE LITIO", "commodity_kw": "Lithium",
        "resource_col": None, "reserve_col": None,
        "allocation": "equal", "is_metallic": False,
        "facility_types": "Mine|Processing|SX-EW|Plant",
    },
    "Lithium_Hydroxide": {
        "col": "COCHILCO_LIOH_2024_MT", "unit": "MT",
        "pattern": "HIDRÓXIDO DE LITIO", "commodity_kw": "Lithium",
        "resource_col": None, "reserve_col": None,
        "allocation": "equal", "is_metallic": False,
        "facility_types": "Mine|Processing|SX-EW|Plant",
    },
    "Lithium_Sulfate": {
        "col": "COCHILCO_LISO4_2024_MT", "unit": "MT",
        "pattern": "SULFATO DE LITIO", "commodity_kw": "Lithium",
        "resource_col": None, "reserve_col": None,
        "allocation": "equal", "is_metallic": False,
        "facility_types": "Mine|Processing|SX-EW|Plant",
    },
    "Iodine": {
        "col": "COCHILCO_IO_2024_MT", "unit": "MT",
        "pattern": "YODO", "commodity_kw": "Iodine",
        "resource_col": None, "reserve_col": None,
        "allocation": "equal", "is_metallic": False,
        "facility_types": "Mine|Processing|Plant",
    },
    "Nitrates": {
        "col": "COCHILCO_NO3_2024_MT", "unit": "MT",
        "pattern": "NITRATOS", "commodity_kw": "Nitrate",
        "resource_col": None, "reserve_col": None,
        "allocation": "equal", "is_metallic": False,
        "facility_types": "Mine|Processing|Plant",
    },
    "Boron_Ulexite": {
        "col": "COCHILCO_ULEXITE_2024_MT", "unit": "MT",
        "pattern": "ULEXITA", "commodity_kw": "Boron",
        "resource_col": None, "reserve_col": None,
        "allocation": "equal", "is_metallic": False,
        "facility_types": "Mine|Processing|Plant",
    },
    "Boric_Acid": {
        "col": "COCHILCO_BORICACID_2024_MT", "unit": "MT",
        "pattern": "ÁCIDO BÓRICO", "commodity_kw": "Boron",
        "resource_col": None, "reserve_col": None,
        "allocation": "equal", "is_metallic": False,
        "facility_types": "Mine|Processing|Plant",
    },
    "Potash": {
        "col": "COCHILCO_KCL_2024_MT", "unit": "MT",
        "pattern": "CLORURO DE POTASIO", "commodity_kw": "Potassium",
        "resource_col": None, "reserve_col": None,
        "allocation": "equal", "is_metallic": False,
        "facility_types": "Mine|Processing|Plant",
    },
    "Salt": {
        "col": "COCHILCO_SALT_2024_MT", "unit": "MT",
        "pattern": "CLORURO DE SODIO", "commodity_kw": "Salt",
        "resource_col": None, "reserve_col": None,
        "allocation": "equal", "is_metallic": False,
        "facility_types": "Mine|Processing|Plant",
    },
    "Copper_Sulfate": {
        "col": "COCHILCO_CUSO4_2024_MT", "unit": "MT",
        "pattern": "SULFATO DE COBRE", "commodity_kw": "Copper Sulfate",
        "resource_col": None, "reserve_col": None,
        "allocation": "equal", "is_metallic": False,
        "facility_types": "Mine|Processing|Plant|SX-EW",
    },
    # Bulk domestic minerals
    "Limestone": {
        "col": "COCHILCO_LIMESTONE_2024_MT", "unit": "MT",
        "pattern": "CALIZA", "commodity_kw": "Limestone",
        "resource_col": None, "reserve_col": None,
        "allocation": "equal", "is_metallic": False,
        "facility_types": "Mine|Quarry|Processing|Plant",
    },
    "Gypsum": {
        "col": "COCHILCO_GYPSUM_2024_MT", "unit": "MT",
        "pattern": "YESO", "commodity_kw": "Gypsum",
        "resource_col": None, "reserve_col": None,
        "allocation": "equal", "is_metallic": False,
        "facility_types": "Mine|Quarry|Processing|Plant",
    },
    "Pumicite": {
        "col": "COCHILCO_PUMICITE_2024_MT", "unit": "MT",
        "pattern": "PUMICITA", "commodity_kw": "Pumicite",
        "resource_col": None, "reserve_col": None,
        "allocation": "equal", "is_metallic": False,
        "facility_types": "Mine|Quarry|Processing|Plant",
    },
    "Quartz": {
        "col": "COCHILCO_QUARTZ_2024_MT", "unit": "MT",
        "pattern": "CUARZO", "commodity_kw": "Silica",
        "resource_col": None, "reserve_col": None,
        "allocation": "equal", "is_metallic": False,
        "facility_types": "Mine|Quarry|Processing|Plant",
    },
    "Silica_Sand": {
        "col": "COCHILCO_SILICASAND_2024_MT", "unit": "MT",
        "pattern": "ARENA SILÍCEA", "commodity_kw": "Silica",
        "resource_col": None, "reserve_col": None,
        "allocation": "equal", "is_metallic": False,
        "facility_types": "Mine|Quarry|Processing|Plant",
    },
    "Bauxitic_Clay": {
        "col": "COCHILCO_BAUXCLAY_2024_MT", "unit": "MT",
        "pattern": "ARCILLA BAUXÍTICA", "commodity_kw": "Bauxitic Clay",
        "resource_col": None, "reserve_col": None,
        "allocation": "equal", "is_metallic": False,
        "facility_types": "Mine|Quarry|Processing|Plant",
    },
    "Kaolin": {
        "col": "COCHILCO_KAOLIN_2024_MT", "unit": "MT",
        "pattern": "CAOLÍN", "commodity_kw": "Kaolin",
        "resource_col": None, "reserve_col": None,
        "allocation": "equal", "is_metallic": False,
        "facility_types": "Mine|Quarry|Processing|Plant",
    },
    "Bentonite": {
        "col": "COCHILCO_BENTONITE_2024_MT", "unit": "MT",
        "pattern": "BENTONITA", "commodity_kw": "Bentonite",
        "resource_col": None, "reserve_col": None,
        "allocation": "equal", "is_metallic": False,
        "facility_types": "Mine|Quarry|Processing|Plant",
    },
    "Diatomite": {
        "col": "COCHILCO_DIATOMITE_2024_MT", "unit": "MT",
        "pattern": "DIATOMITA", "commodity_kw": "Diatomite",
        "resource_col": None, "reserve_col": None,
        "allocation": "equal", "is_metallic": False,
        "facility_types": "Mine|Quarry|Processing|Plant",
    },
    "Dolomite": {
        "col": "COCHILCO_DOLOMITE_2024_MT", "unit": "MT",
        "pattern": "DOLOMITA", "commodity_kw": "Dolomite",
        "resource_col": None, "reserve_col": None,
        "allocation": "equal", "is_metallic": False,
        "facility_types": "Mine|Quarry|Processing|Plant",
    },
    "Talc": {
        "col": "COCHILCO_TALC_2024_MT", "unit": "MT",
        "pattern": "TALCO", "commodity_kw": "Talc",
        "resource_col": None, "reserve_col": None,
        "allocation": "equal", "is_metallic": False,
        "facility_types": "Mine|Quarry|Processing|Plant",
    },
    "Perlite": {
        "col": "COCHILCO_PERLITE_2024_MT", "unit": "MT",
        "pattern": "PERLITA", "commodity_kw": "Perlite",
        "resource_col": None, "reserve_col": None,
        "allocation": "equal", "is_metallic": False,
        "facility_types": "Mine|Quarry|Processing|Plant",
    },
    "Peat": {
        "col": "COCHILCO_PEAT_2024_MT", "unit": "MT",
        "pattern": "TURBA", "commodity_kw": "Peat",
        "resource_col": None, "reserve_col": None,
        "allocation": "equal", "is_metallic": False,
        "facility_types": "Mine|Quarry|Processing|Plant",
    },
    "Phosphate_Rocks": {
        "col": "COCHILCO_PHOSPHATE_2024_MT", "unit": "MT",
        "pattern": "ROCAS FOSFÓRICAS", "commodity_kw": "Phosphate",
        "resource_col": None, "reserve_col": None,
        "allocation": "equal", "is_metallic": False,
        "facility_types": "Mine|Quarry|Processing|Plant",
    },
    "Coquina": {
        "col": "COCHILCO_COQUINA_2024_MT", "unit": "MT",
        "pattern": "COQUINA", "commodity_kw": "Coquina",
        "resource_col": None, "reserve_col": None,
        "allocation": "equal", "is_metallic": False,
        "facility_types": "Mine|Quarry|Processing|Plant",
    },
    "White_CaCO3": {
        "col": "COCHILCO_WHITITE_2024_MT", "unit": "MT",
        "pattern": "CARBONATO DE CALCIO BLANCO", "commodity_kw": "Calcium Carbonate",
        "resource_col": None, "reserve_col": None,
        "allocation": "equal", "is_metallic": False,
        "facility_types": "Mine|Quarry|Processing|Plant",
    },
    "Zeolite": {
        "col": "COCHILCO_ZEOLITE_2024_MT", "unit": "MT",
        "pattern": "ZEOLITA", "commodity_kw": "Zeolite",
        "resource_col": None, "reserve_col": None,
        "allocation": "equal", "is_metallic": False,
        "facility_types": "Mine|Quarry|Processing|Plant",
    },
}

# Initialize production columns — safe to re-run; existing values are not overwritten.

for cfg in MINERAL_COLS.values():
    if cfg["col"] not in inv.columns:
        inv[cfg["col"]] = np.nan

# Parse regional production from C_* sheets

REGION_SHEET_MAP = {
    "C_AricaParinacota": ["Arica", "Parinacota"],
    "C_Tarapaca": ["Tarapacá", "Tarapaca"],
    "C_Antofagasta": ["Antofagasta"],
    "C_Atacama": ["Atacama"],
    "C_Coquimbo": ["Coquimbo"],
    "C_Valparaiso": ["Valparaíso", "Valparaiso"],
    "C_Santiago": ["Metropolitana", "Santiago", "Región Metropolitana"],
    "C_OHiggins": ["O'Higgins", "OHiggins", "Libertador"],
    "C_Maule": ["Maule"],
    "C_BioBio": ["Biobío", "BioBio", "Bio-Bio"],
    "C_Araucania": ["Araucanía", "Araucania"],
    "C_LosLagos": ["Los Lagos"],
    "C_Aysen": ["Aysén", "Aysen", "Ibáñez"],
    "C_Magallanes": ["Magallanes"],
}

wb_path = openpyxl.load_workbook(COCHILCO_PATH, read_only=True, data_only=True)

# Unit detection and conversion
# Metallic minerals have unit hints in the label (e.g. "ORO (Kg de fino)").
# Non-metallic minerals have no unit in the label; the section header
# "II. MINERÍA NO METÁLICA (TM)" defines all as metric tonnes.

def _detect_label_unit(label):
    """Parse the unit from a COCHILCO row label like 'PLATA (TM de fino)'."""
    label_upper = label.upper()
    if "MILES DE TM" in label_upper or "MILES TM" in label_upper:
        return "kMT"
    if "KG" in label_upper:
        return "Kg"
    if "TM" in label_upper:
        return "MT"
    return None

_UNIT_CONVERSIONS = {
    ("MT", "Kg"):   1000.0,
    ("Kg", "MT"):   0.001,
    ("kMT", "MT"):  1000.0,
    ("MT", "kMT"):  0.001,
    ("kMT", "Kg"):  1_000_000.0,
    ("Kg", "kMT"):  1e-6,
}

def _convert_unit(val, source_unit, target_unit):
    if source_unit == target_unit or source_unit is None:
        return val
    factor = _UNIT_CONVERSIONS.get((source_unit, target_unit))
    if factor is None:
        print(f"  WARNING: no conversion for {source_unit} -> {target_unit}, value {val} stored as-is")
        return val
    return val * factor


# Build a lookup: pattern -> mineral name (longest patterns first)
# Sorting by length descending ensures "CARBONATO DE CALCIO BLANCO" matches
# before "CARBONATO DE CALCIO" and "ÁCIDO BÓRICO" before "COMPUESTOS DE BORO".

_PATTERN_LOOKUP = sorted(
    [(cfg["pattern"], mineral) for mineral, cfg in MINERAL_COLS.items()],
    key=lambda x: len(x[0]),
    reverse=True,
)

def _match_mineral(label_upper):
    """Return mineral name if pattern matches at a word boundary, else None.
    Word boundary = start of string, space, '(', '/', or tab before the pattern.
    Prevents 'ORO' from matching inside 'COMPUESTOS DE BORO', etc."""
    for pattern, mineral in _PATTERN_LOOKUP:
        pos = label_upper.find(pattern)
        if pos < 0:
            continue
        if pos > 0 and label_upper[pos - 1] not in (" ", "(", "/", "\t"):
            continue
        return mineral
    return None



regional_production = {}  # {mineral: {sheet_name: value}}

for sheet_name in wb_path.sheetnames:
    if not sheet_name.startswith("C_") or sheet_name == "C0_Cu_Regional_Product":
        continue
    ws = wb_path[sheet_name]
    rows = list(ws.iter_rows(values_only=True))

    # Auto-detect year header row
    yr_col = None
    yr_header_row = None
    for ri in range(min(10, len(rows))):
        row_candidate = rows[ri]
        year_count = sum(1 for v in row_candidate
                         if isinstance(v, (int, float)) and 2004 < v < 2035)
        if year_count >= 3:
            yr_header_row = ri
            for j, v in enumerate(row_candidate):
                if isinstance(v, (int, float)) and int(v) == latest_yr:
                    yr_col = j
                    break
            break
    if yr_col is None:
        print(f"  WARNING: Could not find year {latest_yr} in {sheet_name}")
        continue

    data_start = (yr_header_row + 1) if yr_header_row is not None else 4
    for i in range(data_start, min(data_start + 70, len(rows))):
        row = rows[i]
        label = str(row[0]).upper() if row[0] else ""
        val = row[yr_col] if yr_col < len(row) else None
        if not isinstance(val, (int, float)) or val <= 0:
            continue

        mineral = _match_mineral(label)
        if mineral is None:
            continue

        cfg = MINERAL_COLS[mineral]

        # Unit handling: metallic minerals have units in label; non-metallics are all MT
        if cfg["is_metallic"]:
            # COCHILCO metallic rows always carry a unit label (e.g. "ORO (Kg de fino)").
            # Rows without one are sub-totals or section headers — skip them.
            if not any(u in label for u in ["TM", "KG", "MILES"]):
                continue
            source_unit = _detect_label_unit(label)
        else:
            source_unit = "MT"

        target_unit = cfg["unit"]
        converted = _convert_unit(val, source_unit, target_unit)
        if source_unit is not None and source_unit != target_unit:
            print(f"  Unit conversion [{sheet_name}] {mineral}: "
                  f"{val:,.1f} {source_unit} -> {converted:,.1f} {target_unit}")

        regional_production.setdefault(mineral, {})[sheet_name] = converted

wb_path.close()

# Sanity check: regional sheets sometimes include sub-totals alongside
# mine rows, causing the regional sum to exceed the national figure.
# A ratio above 1.3x almost always indicates double-captured rows.

nat_check = pd.read_excel(COCHILCO_PATH, sheet_name="A_National_Production",
                          header=3, index_col=0)
nat_check.columns = [int(c) if isinstance(c, (int, float)) else c
                      for c in nat_check.columns]

national_totals = {}  # {mineral: value} for summary table

for mineral, cfg in MINERAL_COLS.items():
    reg_total = sum(regional_production.get(mineral, {}).values())
    if reg_total <= 0:
        continue
    nat_val = None
    for idx_label in nat_check.index:
        if cfg["pattern"] in str(idx_label).upper():
            nat_val = nat_check.loc[idx_label, latest_yr] \
                if latest_yr in nat_check.columns else None
            break
    if nat_val is not None and nat_val > 0:
        national_totals[mineral] = nat_val
        ratio = reg_total / nat_val
        if ratio > 1.3:
            print(f"  WARNING: {mineral} regional sum ({reg_total:,.1f} {cfg['unit']}) "
                  f"is {ratio:.1f}x national total ({nat_val:,.1f} {cfg['unit']}). "
                  f"Possible non-production rows captured.")



def _region_mask(region_patterns):
    mask = pd.Series(False, index=inv.index)
    for pat in region_patterns:
        mask |= inv["REGION"].str.contains(pat, case=False, na=False)
    return mask


def _compute_weights_resource_reserve(candidates, cfg):
    """Weights from resource/reserve tonnage. Mines with neither get weight 0."""
    res_col = cfg["resource_col"]
    rev_col = cfg["reserve_col"]
    weights = np.zeros(len(candidates))
    for i, idx in enumerate(candidates.index):
        res_val = inv.at[idx, res_col] if res_col and res_col in inv.columns else np.nan
        if pd.notna(res_val) and res_val > 0:
            weights[i] = res_val
            continue
        rev_val = inv.at[idx, rev_col] if rev_col and rev_col in inv.columns else np.nan
        if pd.notna(rev_val) and rev_val > 0:
            weights[i] = rev_val
    return weights


def _compute_weights_equal(candidates):
    """Equal weights for all candidates."""
    return np.ones(len(candidates))



allocation_summary = {}

for mineral, cfg in MINERAL_COLS.items():
    col_name = cfg["col"]
    unit = cfg["unit"]
    commodity_kw = cfg["commodity_kw"]
    allocation = cfg["allocation"]
    facility_re = cfg.get("facility_types", "Mine.*active")
    regions = regional_production.get(mineral, {})

    if not regions:
        continue

    total_regional = sum(regions.values())
    print(f"\n  {mineral}: {len(regions)} producing region(s), "
          f"total {total_regional:,.1f} {unit}")

    total_assigned = 0.0
    for sheet_name, regional_val in regions.items():
        region_patterns = REGION_SHEET_MAP.get(sheet_name, [])
        if not region_patterns:
            continue

        rmask = _region_mask(region_patterns)

        commodity_mask = inv[comm_col].str.contains(
            commodity_kw, case=False, na=False)

        # For metallic minerals, restrict to active mines
        # For non-metallics, accept mines, processing plants, etc.
        if cfg["is_metallic"]:
            type_mask = inv["FACILITY_TYPE"].str.contains(
                "Mine.*active", case=True, na=False, regex=True)
        else:
            type_mask = inv["FACILITY_TYPE"].str.contains(
                facility_re, case=False, na=False, regex=True)

        candidates = inv[rmask & commodity_mask & type_mask & pd.isna(inv[col_name])]
        region_label = sheet_name.replace("C_", "")

        if len(candidates) == 0:
            # Fallback: drop the facility-type filter. Some legitimate producers
            # (e.g. tailings reprocessors) do not match the default Mine.*active regex.
            candidates = inv[rmask & commodity_mask & pd.isna(inv[col_name])]

        if len(candidates) == 0:
            print(f"    {region_label:<25} {regional_val:>12,.1f} {unit}  "
                  f"-> NO matching facilities")
            continue

        if allocation == "resource_reserve":
            weights = _compute_weights_resource_reserve(candidates, cfg)
            valid_mask = weights > 0
            if not valid_mask.any():
                skipped = ", ".join(inv.at[idx, "FACILITY_NAME"]
                                   for idx in candidates.index)
                print(f"    {region_label:<25} {regional_val:>12,.1f} {unit}  "
                      f"-> SKIPPED (no resource/reserve): {skipped}")
                continue
            candidates = candidates[valid_mask]
            weights = weights[valid_mask]
        else:
            weights = _compute_weights_equal(candidates)

        weight_sum = weights.sum()
        detail_parts = []
        for i_c, idx in enumerate(candidates.index):
            share = regional_val * (weights[i_c] / weight_sum)
            inv.at[idx, col_name] = share
            add_commodity(idx, commodity_kw, inv, comm_col)
            total_assigned += share

            name = inv.at[idx, "FACILITY_NAME"]
            pct = weights[i_c] / weight_sum * 100
            if allocation == "resource_reserve":
                res_col = cfg["resource_col"]
                res_val = inv.at[idx, res_col] if res_col and res_col in inv.columns else np.nan
                src = "resource" if (pd.notna(res_val) and res_val > 0) else "reserve"
                detail_parts.append(f"{name}({src},{pct:.0f}%)")
            else:
                detail_parts.append(f"{name}({pct:.0f}%)")

        detail_str = ", ".join(detail_parts)
        if len(detail_str) > 120:
            detail_str = detail_str[:117] + "..."
        print(f"    {region_label:<25} {regional_val:>12,.1f} {unit}  "
              f"-> {len(candidates)} facilities: {detail_str}")

    pct = (total_assigned / total_regional * 100) if total_regional > 0 else 0
    allocation_summary[mineral] = {
        "n_facilities": inv[col_name].notna().sum(),
        "assigned": total_assigned,
        "regional_total": total_regional,
        "coverage_pct": pct,
        "unit": unit,
    }
    print(f"    TOTAL assigned: {total_assigned:,.1f} / "
          f"{total_regional:,.1f} {unit} ({pct:.1f}%)")


section_header("EXTENDED PRODUCTION MATCHING SUMMARY")

# Also capture national totals for minerals not found in any C_* sheet;
# these go into the pipeline state for the summary table in Cell 9.
all_nat_rows = {}
for idx_label in nat_check.index:
    val = nat_check.loc[idx_label, latest_yr] \
        if latest_yr in nat_check.columns else None
    if isinstance(val, (int, float)) and val > 0:
        all_nat_rows[str(idx_label).strip()] = val

print(f"\n{'Mineral':<25} {'Facilities':>10} {'Assigned':>15} {'National':>15} "
      f"{'Coverage':>8} {'Allocation':>15}")
print("-" * 95)

for mineral, cfg in MINERAL_COLS.items():
    col = cfg["col"]
    n = inv[col].notna().sum()
    total_inv = inv[col].sum() if n > 0 else 0
    nat = national_totals.get(mineral, 0)
    pct = (total_inv / nat * 100) if nat > 0 else 0
    unit = cfg["unit"]
    alloc = cfg["allocation"]
    if n > 0:
        print(f"  {mineral:<25} {n:>8}   {total_inv:>13,.1f} {unit:<4} "
              f"{nat:>13,.1f} {pct:>7.1f}%  {alloc}")

# Molybdenum (matched in Cell 3, not here)
if "COCHILCO_MO_2024_MT" in inv.columns:
    n_mo = inv["COCHILCO_MO_2024_MT"].notna().sum()
    total_mo = inv["COCHILCO_MO_2024_MT"].sum()
    print(f"  {'Molybdenum':<25} {n_mo:>8}   {total_mo:>13,.1f} MT   "
          f"{'38486.9':>13} {'98.5':>7}%  company")


_national_production = {}
for mineral, cfg in MINERAL_COLS.items():
    _national_production[mineral] = {
        "national_total": national_totals.get(mineral, 0),
        "unit": cfg["unit"],
        "col": cfg["col"],
        "commodity_kw": cfg["commodity_kw"],
        "allocation": cfg["allocation"],
        "assigned": inv[cfg["col"]].sum() if inv[cfg["col"]].notna().any() else 0,
    }




## 2F: Extended Commodity Pricing & Monetary Valuation

Parses 2024 commodity prices from the COCHILCO Anuario (Tables 96-98),
adds implied FOB unit values from export data (Tabla 7/9), and includes
USGS benchmark prices for domestic-only minerals. Computes USD production
values for all minerals with matched facility production.


In [ ]:

# 2F. COMMODITY PRICE EXTRACTION & MONETARY VALUATION
#
# Reads 2024 prices from the COCHILCO Anuario via commodity_prices_2024.py,
# then adds implied FOB prices from export data and manual benchmarks for
# domestic-only minerals. Multiplies each facility's production column by
# the corresponding unit price to create USD value columns.
#
# Price sources (in priority order):
#   1. Tabla 96  -> Copper (LME Grade A Settlement, cents US$/lb -> USD/MT)
#   2. Tabla 97  -> Mo, Zn, Au, Ag, Pb (various exchanges -> USD/MT or USD/kg)
#   3. Tabla 98  -> Lithium Carbonate, Iodine, Boric Acid (midpoint of min/max)
#   4. Implied   -> FOB unit values from Tabla 7 (values) / Tabla 9 (volumes)
#   5. Manual    -> USGS/Platts benchmarks for domestic-only minerals

section_header("2F. COMMODITY PRICE EXTRACTION")

from commodity_prices_2024 import build_prices

prices_mt, prices_kg, price_units, price_sources, price_bands = \
    build_prices(anuario_path=COCHILCO_ORIG)

# Prices for minerals absent from COCHILCO Tablas 96-98 (derived sources below):
#
# Iron ore: Platts IODEX 62% Fe CFR China 2024 annual avg
# Salt: implied from Tabla 7 ($140.6M) / Tabla 9 (6,400 kMT)
# Sodium Nitrate: implied from Tabla 7 ($52.42M) / Tabla 9 (41,657 MT)
# Potassium Nitrate: implied from Tabla 7 ($4.6M) / Tabla 9 (7,049 MT)
# Borates (ulexite): implied from Tabla 7 ($9.4M) / Tabla 9 (31,176 MT)
# Potash (KCl): USGS avg 2024 ~$260/MT (FOB)
# Others: USGS Mineral Commodity Summaries 2024 avg unit values.

# Tier 4 benchmarks for minerals absent from COCHILCO Tablas 96-98.
# Each MANUAL_PRICES entry is (price_USD_per_MT, source_citation).
for mineral, (price, source) in MANUAL_PRICES.items():
    if price is not None:      # skip None placeholders (Li compounds — handled below)
        prices_mt[mineral] = price
        price_units[mineral] = "USD/MT"
        price_sources[mineral] = source

# LiCO3 price comes from COCHILCO Tabla 98. No Tabla equivalent exists for
# LiOH or Li2SO4, so prices are derived from market ratios to LiCO3.
# LiOH traded at ~1.1x LiCO3 in 2024 (battery-grade premium); Li2SO4 at
# ~0.7x (lower purity, smaller market). Ratios from LITHIUM_PRICE_MULTIPLIERS.

lico3_price = prices_mt.get("Lithium", None)
if lico3_price:
    # Prices for Li sub-compounds derived via LITHIUM_PRICE_MULTIPLIERS (pipeline_constants.py)
    for li_form, multiplier in LITHIUM_PRICE_MULTIPLIERS.items():
        prices_mt[li_form] = lico3_price * multiplier
        price_units[li_form] = "USD/MT"
        src_tag = "COCHILCO Tabla 98" if multiplier == 1.0 else (
            f"~{multiplier:.2f}x LiCO3 midpoint (LITHIUM_PRICE_MULTIPLIERS, pipeline_constants.py)")
        price_sources[li_form] = src_tag

    # Aggregate price for national summary (not used in facility VALUATION_MAP)
    prices_mt["Lithium Compounds"] = lico3_price
    price_units["Lithium Compounds"] = "USD/MT"
    price_sources["Lithium Compounds"] = "Approx. LiCO3 price (dominant compound, aggregate)"


print(f"{'Mineral':<25} {'Price':>12}  {'Unit':<8}  Source")
print("-" * 100)
for mineral in sorted(set(list(prices_mt.keys()) + list(prices_kg.keys()))):
    p = prices_mt.get(mineral) or prices_kg.get(mineral)
    u = price_units.get(mineral, "")
    s = price_sources.get(mineral, "")
    tag = " [band]" if mineral in price_bands else ""
    print(f"  {mineral:<25} {p:>12,.2f}  {u:<8}  {s}{tag}")


# Define production-to-price mapping
# Each entry: (production_col, price_key, unit_multiplier, value_col, price_type)
#   unit_multiplier converts the production column unit into the price unit.
#   kMT -> MT requires *1000; MT -> MT and kg -> kg require *1.

VALUATION_MAP = [
    # Metallic minerals (original 6)
    ("COCHILCO_CU_2024_KMT",  "Copper",             1000.0, "USD_VALUE_CU",         "mt"),
    ("COCHILCO_MO_2024_MT",   "Molybdenum",           1.0,  "USD_VALUE_MO",         "mt"),
    ("COCHILCO_AU_2024_KG",   "Gold",                 1.0,  "USD_VALUE_AU",         "kg"),
    ("COCHILCO_AG_2024_KG",   "Silver",               1.0,  "USD_VALUE_AG",         "kg"),
    ("COCHILCO_FE_2024_KMT",  "Iron",              1000.0,  "USD_VALUE_FE",         "mt"),
    ("COCHILCO_ZN_2024_MT",   "Zinc",                 1.0,  "USD_VALUE_ZN",         "mt"),
    ("COCHILCO_PB_2024_MT",   "Lead",                 1.0,  "USD_VALUE_PB",         "mt"),
    # Lithium compounds
    ("COCHILCO_LICO3_2024_MT",     "Lithium Carbonate",  1.0, "USD_VALUE_LICO3",    "mt"),
    ("COCHILCO_LIOH_2024_MT",      "Lithium Hydroxide",  1.0, "USD_VALUE_LIOH",     "mt"),
    ("COCHILCO_LISO4_2024_MT",     "Lithium Sulfate",    1.0, "USD_VALUE_LISO4",    "mt"),
    # Other high-value non-metallics
    ("COCHILCO_IO_2024_MT",        "Iodine",             1.0, "USD_VALUE_IO",       "mt"),
    ("COCHILCO_NO3_2024_MT",       "Sodium Nitrate",     1.0, "USD_VALUE_NO3",      "mt"),
    ("COCHILCO_ULEXITE_2024_MT",   "Ulexite",            1.0, "USD_VALUE_ULEXITE",  "mt"),
    ("COCHILCO_BORICACID_2024_MT", "Boric Acid",         1.0, "USD_VALUE_BORICACID","mt"),
    ("COCHILCO_KCL_2024_MT",       "Potash",             1.0, "USD_VALUE_KCL",      "mt"),
    ("COCHILCO_SALT_2024_MT",      "Salt",               1.0, "USD_VALUE_SALT",     "mt"),
    ("COCHILCO_CUSO4_2024_MT",     "Copper Sulfate",     1.0, "USD_VALUE_CUSO4",    "mt"),
    # Bulk domestic minerals
    ("COCHILCO_LIMESTONE_2024_MT", "Limestone",          1.0, "USD_VALUE_LIMESTONE", "mt"),
    ("COCHILCO_COQUINA_2024_MT",   "Coquina",            1.0, "USD_VALUE_COQUINA",  "mt"),
    ("COCHILCO_WHITITE_2024_MT",   "White CaCO3",        1.0, "USD_VALUE_WHITECACO3","mt"),
    ("COCHILCO_GYPSUM_2024_MT",    "Gypsum",             1.0, "USD_VALUE_GYPSUM",   "mt"),
    ("COCHILCO_PUMICITE_2024_MT",  "Pumicite",           1.0, "USD_VALUE_PUMICITE", "mt"),
    ("COCHILCO_QUARTZ_2024_MT",    "Quartz",             1.0, "USD_VALUE_QUARTZ",   "mt"),
    ("COCHILCO_SILICASAND_2024_MT","Silica Sand",         1.0, "USD_VALUE_SILICASAND","mt"),
    ("COCHILCO_BAUXCLAY_2024_MT",  "Bauxitic Clay",      1.0, "USD_VALUE_BAUXCLAY", "mt"),
    ("COCHILCO_KAOLIN_2024_MT",    "Kaolin",             1.0, "USD_VALUE_KAOLIN",   "mt"),
    ("COCHILCO_BENTONITE_2024_MT", "Bentonite",          1.0, "USD_VALUE_BENTONITE","mt"),
    ("COCHILCO_DIATOMITE_2024_MT", "Diatomite",          1.0, "USD_VALUE_DIATOMITE","mt"),
    ("COCHILCO_DOLOMITE_2024_MT",  "Dolomite",           1.0, "USD_VALUE_DOLOMITE", "mt"),
    ("COCHILCO_TALC_2024_MT",      "Talc",               1.0, "USD_VALUE_TALC",     "mt"),
    ("COCHILCO_PERLITE_2024_MT",   "Perlite",            1.0, "USD_VALUE_PERLITE",  "mt"),
    ("COCHILCO_PEAT_2024_MT",      "Peat",               1.0, "USD_VALUE_PEAT",     "mt"),
    ("COCHILCO_PHOSPHATE_2024_MT", "Phosphate Rocks",    1.0, "USD_VALUE_PHOSPHATE","mt"),
    ("COCHILCO_ZEOLITE_2024_MT",   "Zeolite",            1.0, "USD_VALUE_ZEOLITE",  "mt"),
]

section_header("MONETARY VALUATION")

print(f"{'Mineral':<25} {'Facilities':>10} {'Production':>15} {'Unit':>5}  "
      f"{'Price (USD)':>12} {'Total Value':>18}")
print("-" * 95)

valued_count = 0
for prod_col, price_key, multiplier, val_col, price_type in VALUATION_MAP:
    if prod_col not in inv.columns:
        continue

    price = prices_mt.get(price_key) if price_type == "mt" \
        else prices_kg.get(price_key)
    if price is None:
        continue

    inv[val_col] = inv[prod_col] * multiplier * price

    n = inv[val_col].notna().sum()
    if n == 0:
        continue
    valued_count += 1
    total_prod = inv[prod_col].sum()
    total_val  = inv[val_col].sum()

    unit_label = "kMT" if multiplier == 1000 else \
        ("kg" if price_type == "kg" else "MT")
    print(f"  {price_key:<25} {n:>8}   {total_prod:>13,.1f} {unit_label:>5}  "
          f"{price:>12,.2f}  ${total_val / 1e6:>14,.1f}M")

value_cols = [vc for _, _, _, vc, _ in VALUATION_MAP if vc in inv.columns]
inv["USD_VALUE_TOTAL"] = inv[value_cols].sum(axis=1, min_count=1)

n_valued = inv["USD_VALUE_TOTAL"].notna().sum()
grand_total = inv["USD_VALUE_TOTAL"].sum()
print(f"\nMinerals with at least one valued facility: {valued_count}")
print(f"Facilities with at least one valued mineral: {n_valued}")
print(f"Grand total estimated production value:       ${grand_total / 1e6:,.1f}M USD")

commodity_prices = {
    'prices_mt': prices_mt,
    'prices_kg': prices_kg,
    'price_units': price_units,
    'price_sources': price_sources,
    'price_bands': price_bands,
}



## 2G: National Production Summary

Creates `Chile_National_Production_2024.csv` with every mineral Chile
produces, their estimated USD values, and flags for facility allocation
and supply chain coverage.


In [ ]:

# 2G. National production summary
# Creates Chile_National_Production_2024.csv covering all ~35 minerals Chile
# produces. For each mineral: volume, price, estimated USD value, and flags
# indicating whether facility-level allocation succeeded and whether supply
# chain edges exist in Notebook B. The national summary captures minerals
# that could not be allocated to individual facilities (e.g. aggregates like
# "Lithium Compounds") and is the primary input for the final visualisation.

section_header("2G. NATIONAL PRODUCTION SUMMARY TABLE")

nat_prod = pd.read_excel(COCHILCO_PATH, sheet_name="A_National_Production",
                         header=3, index_col=0)
nat_prod.columns = [int(c) if isinstance(c, (int, float)) else c
                     for c in nat_prod.columns]

# Minerals with supply chain edges (Notebook B)
SC_MINERALS = {
    "Copper", "Molybdenum", "Gold", "Silver", "Iron", "Zinc",
    "Lithium", "Iodine", "Rhenium", "Boron", "Nitrate", "Sulfuric Acid",
}

summary_rows = []

# Map national production labels to our mineral names and units
NAT_LABEL_MAP = {
    "COBRE": ("Copper", "kMT", "COCHILCO_CU_2024_KMT"),
    "MOLIBDENO": ("Molybdenum", "MT", "COCHILCO_MO_2024_MT"),
    "ORO": ("Gold", "Kg", "COCHILCO_AU_2024_KG"),
    "PLATA": ("Silver", "Kg", "COCHILCO_AG_2024_KG"),
    "HIERRO": ("Iron", "kMT", "COCHILCO_FE_2024_KMT"),
    "ZINC": ("Zinc", "MT", "COCHILCO_ZN_2024_MT"),
    "PLOMO": ("Lead", "MT", "COCHILCO_PB_2024_MT"),
    "MANGANESO": ("Manganese", "MT", None),
    "COMPUESTOS DE LITIO": ("Lithium Compounds", "MT", None),
    "CARBONATO DE LITIO": ("Lithium Carbonate", "MT", "COCHILCO_LICO3_2024_MT"),
    "HIDRÓXIDO DE LITIO": ("Lithium Hydroxide", "MT", "COCHILCO_LIOH_2024_MT"),
    "SULFATO DE LITIO": ("Lithium Sulfate", "MT", "COCHILCO_LISO4_2024_MT"),
    "YODO": ("Iodine", "MT", "COCHILCO_IO_2024_MT"),
    "NITRATOS": ("Nitrates", "MT", "COCHILCO_NO3_2024_MT"),
    "ULEXITA": ("Boron (Ulexite)", "MT", "COCHILCO_ULEXITE_2024_MT"),
    "ÁCIDO BÓRICO": ("Boric Acid", "MT", "COCHILCO_BORICACID_2024_MT"),
    "CLORURO DE POTASIO": ("Potash (KCl)", "MT", "COCHILCO_KCL_2024_MT"),
    "CLORURO DE SODIO": ("Salt (NaCl)", "MT", "COCHILCO_SALT_2024_MT"),
    "SULFATO DE COBRE": ("Copper Sulfate", "MT", "COCHILCO_CUSO4_2024_MT"),
    "CARBONATO DE CALCIO": ("Calcium Carbonate", "MT", None),
    "CALIZA": ("Limestone", "MT", "COCHILCO_LIMESTONE_2024_MT"),
    "COQUINA": ("Coquina", "MT", "COCHILCO_COQUINA_2024_MT"),
    "YESO": ("Gypsum", "MT", "COCHILCO_GYPSUM_2024_MT"),
    "PUMICITA": ("Pumicite", "MT", "COCHILCO_PUMICITE_2024_MT"),
    "RECURSOS SILÍCEOS": ("Silica Ores", "MT", None),
    "CUARZO": ("Quartz", "MT", "COCHILCO_QUARTZ_2024_MT"),
    "ARENA SILÍCEA": ("Silica Sand", "MT", "COCHILCO_SILICASAND_2024_MT"),
    "ARCILLAS": ("Clay", "MT", None),
    "ARCILLA BAUXÍTICA": ("Bauxitic Clay", "MT", "COCHILCO_BAUXCLAY_2024_MT"),
    "CAOLÍN": ("Kaolin", "MT", "COCHILCO_KAOLIN_2024_MT"),
    "BENTONITA": ("Bentonite", "MT", "COCHILCO_BENTONITE_2024_MT"),
    "DIATOMITA": ("Diatomite", "MT", "COCHILCO_DIATOMITE_2024_MT"),
    "DOLOMITA": ("Dolomite", "MT", "COCHILCO_DOLOMITE_2024_MT"),
    "TALCO": ("Talc", "MT", "COCHILCO_TALC_2024_MT"),
    "PERLITA": ("Perlite", "MT", "COCHILCO_PERLITE_2024_MT"),
    "TURBA": ("Peat", "MT", "COCHILCO_PEAT_2024_MT"),
    "ROCAS FOSFÓRICAS": ("Phosphate Rocks", "MT", "COCHILCO_PHOSPHATE_2024_MT"),
    "ZEOLITA": ("Zeolite", "MT", "COCHILCO_ZEOLITE_2024_MT"),
}

# Sort patterns longest-first to prevent short patterns matching inside
# longer words (e.g. "ORO" inside "COMPUESTOS DE BORO")
_NAT_SORTED = sorted(NAT_LABEL_MAP.items(), key=lambda x: len(x[0]), reverse=True)

# Reverse lookup for price keys
PRICE_KEY_MAP = {
    "Copper": "Copper", "Molybdenum": "Molybdenum",
    "Gold": "Gold", "Silver": "Silver",
    "Iron": "Iron", "Zinc": "Zinc", "Lead": "Lead",
    "Lithium Compounds": "Lithium Compounds",
    "Lithium Carbonate": "Lithium Carbonate",
    "Lithium Hydroxide": "Lithium Hydroxide",
    "Lithium Sulfate": "Lithium Sulfate",
    "Iodine": "Iodine", "Nitrates": "Sodium Nitrate",
    "Boron (Ulexite)": "Ulexite", "Boric Acid": "Boric Acid",
    "Potash (KCl)": "Potash", "Salt (NaCl)": "Salt",
    "Copper Sulfate": "Copper Sulfate",
    "Calcium Carbonate": "Calcium Carbonate",
    "Limestone": "Limestone", "Coquina": "Coquina",
    "Gypsum": "Gypsum", "Pumicite": "Pumicite",
    "Silica Ores": "Silica Ores", "Quartz": "Quartz",
    "Silica Sand": "Silica Sand", "Clay": "Clay",
    "Bauxitic Clay": "Bauxitic Clay", "Kaolin": "Kaolin",
    "Bentonite": "Bentonite", "Diatomite": "Diatomite",
    "Dolomite": "Dolomite", "Talc": "Talc",
    "Perlite": "Perlite", "Peat": "Peat",
    "Phosphate Rocks": "Phosphate Rocks",
    "Zeolite": "Zeolite",
}

for idx_label in nat_prod.index:
    label_str = str(idx_label).strip()
    label_upper = label_str.upper()

    # Skip section headers and footnotes
    if label_upper.startswith("I.") or label_upper.startswith("II.") or \
       label_upper.startswith("III.") or label_upper.startswith("-") or \
       label_upper.startswith("("):
        continue

    val = nat_prod.loc[idx_label, latest_yr] \
        if latest_yr in nat_prod.columns else None
    if not isinstance(val, (int, float)) or val <= 0:
        continue

    # Match to our mineral name using longest-pattern-first ordering
    # and word-boundary check to prevent "ORO" matching inside "BORO", etc.
    matched_name = None
    matched_unit = "MT"
    matched_col = None
    for pattern, (name, unit, col) in _NAT_SORTED:
        # Word-boundary check: pattern must start at a word boundary
        # (beginning of string, after space, or after "(")
        pos = label_upper.find(pattern)
        if pos < 0:
            continue
        if pos > 0 and label_upper[pos - 1] not in (" ", "(", "/", "\t"):
            continue
        matched_name = name
        matched_unit = unit
        matched_col = col
        break

    if matched_name is None:
        matched_name = label_str.split("/")[0].strip().title()

    price_key = PRICE_KEY_MAP.get(matched_name, matched_name)
    price = prices_mt.get(price_key) or prices_kg.get(price_key)
    price_unit = price_units.get(price_key, "")
    price_src = price_sources.get(price_key, "")

    if price and matched_unit == "kMT":
        est_value = val * 1000 * price
    elif price and matched_unit == "Kg" and "kg" in price_unit.lower():
        est_value = val * price
    elif price and matched_unit == "Kg" and "mt" in price_unit.lower():
        est_value = val / 1000 * price
    elif price:
        est_value = val * price
    else:
        est_value = None

    n_facilities = 0
    assigned_total = 0
    if matched_col and matched_col in inv.columns:
        n_facilities = inv[matched_col].notna().sum()
        assigned_total = inv[matched_col].sum() if n_facilities > 0 else 0

    has_sc = any(sc_kw in matched_name for sc_kw in SC_MINERALS) or \
             matched_name in SC_MINERALS

    summary_rows.append({
        "MINERAL": matched_name,
        "COCHILCO_LABEL": label_str,
        "NATIONAL_PRODUCTION_2024": val,
        "UNIT": matched_unit,
        "USD_PRICE": price,
        "PRICE_UNIT": price_unit,
        "PRICE_SOURCE": price_src,
        "EST_VALUE_USD": est_value,
        "N_FACILITIES_ALLOCATED": n_facilities,
        "PRODUCTION_ALLOCATED": assigned_total,
        "ALLOCATION_COVERAGE_PCT": (assigned_total / val * 100)
            if val > 0 and assigned_total > 0 else 0,
        "HAS_SUPPLY_CHAIN": has_sc,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values("EST_VALUE_USD", ascending=False, na_position="last")

summary_path = os.path.join(DIR_INTERMED, "Chile_National_Production_2024.csv")
summary_df.to_csv(summary_path, index=False)
print(f"Saved: {summary_path} ({len(summary_df)} minerals)")

print(f"\n{'Mineral':<25} {'Production':>15} {'Unit':>5}  "
      f"{'Value (M USD)':>15}  {'Facilities':>10}  {'SC':>3}")
print("-" * 85)
for _, row in summary_df.head(30).iterrows():
    val_str = f"${row['EST_VALUE_USD']/1e6:,.1f}M" if pd.notna(row['EST_VALUE_USD']) else "n/a"
    sc = "Y" if row["HAS_SUPPLY_CHAIN"] else ""
    print(f"  {row['MINERAL']:<25} {row['NATIONAL_PRODUCTION_2024']:>13,.1f} {row['UNIT']:>5}  "
          f"{val_str:>15}  {row['N_FACILITIES_ALLOCATED']:>10}  {sc:>3}")

print(f"\nTotal estimated value: "
      f"${summary_df['EST_VALUE_USD'].sum()/1e6:,.1f}M USD")



## Save State


In [ ]:
# Save state for Notebook B
_state = {
    'inv': inv, 'links': links, 'comm_col': comm_col, 'idle_mines': idle_mines,
    'inv_path': inv_path, 'links_path': links_path,
    'COMPANY_TO_DEPOSIT': COMPANY_TO_DEPOSIT,
    'CODELCO_EXTRA_SEARCH': CODELCO_EXTRA_SEARCH,
    'SMELTERS': SMELTERS, 'PORTS': PORTS,
    'SMELTER_NAME_MAP': SMELTER_NAME_MAP,
    'DEDICATED_PORT': DEDICATED_PORT,
    'CODELCO_CATHODE_ROUTING': CODELCO_CATHODE_ROUTING,
    'MATCH_DISAMBIGUATION': MATCH_DISAMBIGUATION,
    'IRON_MINE_NAMES': IRON_MINE_NAMES,
    'ZINC_MINE_NAMES': ZINC_MINE_NAMES,
    'n_idle_links': n_idle_links,
    'cu_total': cu_total,
    'commodity_prices': commodity_prices,
    'MINERAL_COLS': MINERAL_COLS,
    'national_production': summary_df,
}
save_state(_state, 2)

# Also save the updated inventory CSV
inv.to_csv(inv_path, index=False)
print(f"Inventory saved: {inv_path} ({len(inv)} rows, {len(inv.columns)} cols)")

